# 04 - Silver: SECOP II (Contratos Públicos)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

| | Tabla |
|---|---|
| Entrada | `datos_empresas.bronze.de_semiestructurado_secop_contratos_api` (87 columnas, todas texto) |
| Salida Silver | `datos_empresas.silver.de_secop_contratos` |
| Salida DQL | `datos_empresas.dql.de_secop_contratos_dql` |
| Llave (grano) | `id_contrato` |

## Transformaciones

* **Estandarización de nombres**: las columnas que la API entrega con caracteres corruptos o nombres largos se renombran a `snake_case` legible. Por ejemplo, `liquidaci_n` -> `requiere_liquidacion`, `tipo_de_contrato` -> `tipo_contrato` y `urlproceso` -> `url_proceso` (solo se extrae la URL del JSON).
* **Fechas** `fecha_de_firma`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato`: texto ISO -> `DATE`. `ultima_actualizacion` -> `TIMESTAMP`, que define la versión vigente de cada contrato.
* **Valores monetarios**: texto -> `DECIMAL(22,2)`. Se usa decimal y no `DOUBLE` para no perder precisión en pesos.
* **Indicadores Si/No** (`es_pyme`, `es_grupo`, `espostconflicto`, `liquidaci_n`, `el_contrato_puede_ser_prorrogado`): texto -> `BOOLEAN`.
* **`documento_proveedor_normalizado`**: solo dígitos y sin ceros a la izquierda, para cruzar con RUES en Gold. El texto `No Definido` queda `NULL`.
* **Minimización de datos sensibles**: no pasan a Silver el banco y el número de cuenta del proveedor, ni los documentos del representante legal, del supervisor y de los ordenadores del gasto y de pago. No tienen uso analítico y siguen disponibles en Bronze.

## Reglas de calidad (rechazo hacia DQL)

| # | Regla (`_motivo_rechazo`) | Condición | Por qué es pertinente |
|---|---|---|---|
| 1 | `campo_obligatorio_vacio` | `id_contrato`, `nit_entidad` o `documento_proveedor` vacíos, o el proveedor es `No Definido` | Un contrato sin identificador no se puede rastrear. Sin entidad contratante o sin proveedor adjudicado no se puede responder *quién contrata con quién*, que es la pregunta central del hecho de contratos en Gold. |
| 2 | `valor_contrato_invalido` | `valor_del_contrato` no numérico, nulo o `<= 0` | Regla de negocio: todo contrato público tiene un valor positivo. Los valores cero o negativos sesgarían los totales y promedios de contratación en Gold. |
| 3 | `fechas_inconsistentes` | Falta la fecha de firma, inicio o fin; el inicio es posterior al fin; o la firma es posterior al fin | Un contrato no puede terminar antes de empezar ni firmarse después de terminar. Además, la fecha de firma es la que se usa para convertir el valor a dólares con la TRM. |
| 4 | `valor_pagado_invalido` | `valor_pagado` negativo o mayor que `valor_del_contrato` | No se puede pagar más de lo contratado (las adiciones de valor no vienen en este dataset). Si se aceptara, el indicador `porcentaje_pagado` de Gold quedaría por encima del 100 %. |
| 5 | `duplicado_version_anterior` | Hay una versión más reciente del mismo `id_contrato` | **Unicidad**: la ingesta incremental vuelve a traer un contrato cada vez que se actualiza en SECOP. Se conserva la versión con mayor `ultima_actualizacion`. |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Lectura de Bronze, tipado y estandarización de nombres

In [ ]:
%python
df_bronze = preparar_bronze(spark.table(BRONZE_SECOP))

df_tipado = df_bronze.select(
    "_registro_bronze", "_hash_bronze", "_id_registro_bronze",
    F.col("_ingested_at").alias("_bronze_ingested_at"),
    # Identificación del contrato
    texto_limpio("id_contrato").alias("id_contrato"),
    texto_limpio("referencia_del_contrato").alias("referencia_contrato"),
    texto_limpio("proceso_de_compra").alias("proceso_compra"),
    # Entidad contratante
    texto_limpio("nombre_entidad", mayusculas=True).alias("nombre_entidad"),
    texto_limpio("nit_entidad").alias("nit_entidad"),
    texto_limpio("codigo_entidad").alias("codigo_entidad"),
    texto_limpio("departamento").alias("departamento"),
    texto_limpio("ciudad").alias("ciudad"),
    texto_limpio("orden").alias("orden"),
    texto_limpio("sector").alias("sector"),
    texto_limpio("rama").alias("rama"),
    a_booleano("entidad_centralizada").alias("es_entidad_centralizada"),
    # Características del contrato
    texto_limpio("estado_contrato").alias("estado_contrato"),
    texto_limpio("tipo_de_contrato").alias("tipo_contrato"),
    texto_limpio("modalidad_de_contratacion").alias("modalidad_contratacion"),
    texto_limpio("justificacion_modalidad_de").alias("justificacion_modalidad"),
    texto_limpio("codigo_de_categoria_principal").alias("codigo_categoria_principal"),
    texto_limpio("objeto_del_contrato").alias("objeto_contrato"),
    texto_limpio("descripcion_del_proceso").alias("descripcion_proceso"),
    texto_limpio("origen_de_los_recursos").alias("origen_recursos"),
    texto_limpio("destino_gasto").alias("destino_gasto"),
    a_booleano("espostconflicto").alias("es_postconflicto"),
    a_booleano("liquidaci_n").alias("requiere_liquidacion"),
    a_booleano("el_contrato_puede_ser_prorrogado").alias("puede_prorrogarse"),
    # Proveedor adjudicado
    texto_limpio("tipodocproveedor").alias("tipo_documento_proveedor"),
    F.when(F.lower(F.trim("documento_proveedor")) == "no definido", F.lit(None))
     .otherwise(texto_limpio("documento_proveedor")).alias("documento_proveedor"),
    solo_digitos("documento_proveedor").alias("documento_proveedor_normalizado"),
    texto_limpio("codigo_proveedor").alias("codigo_proveedor"),
    texto_limpio("proveedor_adjudicado", mayusculas=True).alias("proveedor_adjudicado"),
    a_booleano("es_pyme").alias("es_pyme"),
    a_booleano("es_grupo").alias("es_grupo"),
    # Fechas
    fecha_iso("fecha_de_firma").alias("fecha_firma"),
    fecha_iso("fecha_de_inicio_del_contrato").alias("fecha_inicio"),
    fecha_iso("fecha_de_fin_del_contrato").alias("fecha_fin"),
    F.expr(f"try_to_timestamp(ultima_actualizacion, \"{FORMATO_FECHA_ISO}\")").alias("ultima_actualizacion"),
    F.expr("try_cast(dias_adicionados AS INT)").alias("dias_adicionados"),
    # Valores (COP)
    decimal_seguro("valor_del_contrato").alias("valor_contrato"),
    decimal_seguro("valor_de_pago_adelantado").alias("valor_pago_adelantado"),
    decimal_seguro("valor_facturado").alias("valor_facturado"),
    decimal_seguro("valor_pagado").alias("valor_pagado"),
    decimal_seguro("valor_pendiente_de_pago").alias("valor_pendiente_pago"),
    decimal_seguro("valor_pendiente_de_ejecucion").alias("valor_pendiente_ejecucion"),
    # La API entrega {"url": "https://..."}; se conserva solo la URL
    F.regexp_extract("urlproceso", r"(https?://[^\"'\s}]+)", 1).alias("url_proceso"),
)

df_versionado = marcar_version(
    df_tipado, ["id_contrato"],
    [F.col("ultima_actualizacion").desc_nulls_last(), F.col("_bronze_ingested_at").desc()],
)

---
## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Antes de rechazar, se mide sobre Bronze cuántas filas incumpliría cada regla. Una fila puede incumplir varias, así que las columnas no se suman. Resultado con el Bronze del 21-oct-2026 (63.770 filas): `campo_obligatorio_vacio` = 404 (proveedor `No Definido`), `valor_contrato_invalido` = 235, `valor_pagado_invalido` = 24, `fechas_inconsistentes` = 8, duplicados = 0.

In [ ]:
%sql
-- Evidencia: filas de Bronze que incumplirían cada regla (mismas condiciones que el Paso 2)
WITH s AS (
  SELECT *,
         TO_DATE(TRY_TO_TIMESTAMP(fecha_de_firma, "yyyy-MM-dd'T'HH:mm:ss.SSS"))               AS ff,
         TO_DATE(TRY_TO_TIMESTAMP(fecha_de_inicio_del_contrato, "yyyy-MM-dd'T'HH:mm:ss.SSS")) AS fi,
         TO_DATE(TRY_TO_TIMESTAMP(fecha_de_fin_del_contrato, "yyyy-MM-dd'T'HH:mm:ss.SSS"))    AS fe,
         TRY_CAST(valor_del_contrato AS DECIMAL(22,2)) AS v,
         TRY_CAST(valor_pagado AS DECIMAL(22,2))       AS vp,
         ROW_NUMBER() OVER (PARTITION BY TRIM(id_contrato)
                            ORDER BY TRY_TO_TIMESTAMP(ultima_actualizacion, "yyyy-MM-dd'T'HH:mm:ss.SSS") DESC NULLS LAST) AS version
  FROM datos_empresas.bronze.de_semiestructurado_secop_contratos_api
)
SELECT
  COUNT(*) AS filas_bronze,
  SUM(CASE WHEN TRIM(COALESCE(id_contrato, '')) = '' OR TRIM(COALESCE(nit_entidad, '')) = ''
            OR TRIM(COALESCE(documento_proveedor, '')) = '' OR LOWER(TRIM(documento_proveedor)) = 'no definido' THEN 1 ELSE 0 END) AS r1_campo_obligatorio_vacio,
  SUM(CASE WHEN v IS NULL OR v <= 0 THEN 1 ELSE 0 END) AS r2_valor_contrato_invalido,
  SUM(CASE WHEN ff IS NULL OR fi IS NULL OR fe IS NULL OR fi > fe OR ff > fe THEN 1 ELSE 0 END) AS r3_fechas_inconsistentes,
  SUM(CASE WHEN vp < 0 OR vp > v THEN 1 ELSE 0 END) AS r4_valor_pagado_invalido,
  SUM(CASE WHEN version > 1 THEN 1 ELSE 0 END) AS r5_duplicado_version_anterior
FROM s;

---
## Paso 2: Reglas de calidad y separación Silver / DQL

In [ ]:
%python
REGLAS = [
    ("campo_obligatorio_vacio",
     es_vacio("id_contrato") | es_vacio("nit_entidad") | es_vacio("documento_proveedor")),
    ("valor_contrato_invalido",
     F.col("valor_contrato").isNull() | (F.col("valor_contrato") <= 0)),
    ("fechas_inconsistentes",
     F.col("fecha_firma").isNull() | F.col("fecha_inicio").isNull() | F.col("fecha_fin").isNull()
     | (F.col("fecha_inicio") > F.col("fecha_fin"))
     | (F.col("fecha_firma") > F.col("fecha_fin"))),
    ("valor_pagado_invalido",
     F.coalesce((F.col("valor_pagado") < 0) | (F.col("valor_pagado") > F.col("valor_contrato")), F.lit(False))),
    ("duplicado_version_anterior",
     (F.col("_version") > 1) & ~es_vacio("id_contrato")),
]

df_aceptados, df_rechazados = separar_aceptados_rechazados(df_versionado, REGLAS)

COLUMNAS_SILVER = [
    "id_contrato", "referencia_contrato", "proceso_compra",
    "nombre_entidad", "nit_entidad", "codigo_entidad", "departamento", "ciudad", "orden", "sector", "rama",
    "es_entidad_centralizada",
    "estado_contrato", "tipo_contrato", "modalidad_contratacion", "justificacion_modalidad",
    "codigo_categoria_principal", "objeto_contrato", "descripcion_proceso", "origen_recursos", "destino_gasto",
    "es_postconflicto", "requiere_liquidacion", "puede_prorrogarse",
    "tipo_documento_proveedor", "documento_proveedor", "documento_proveedor_normalizado", "codigo_proveedor",
    "proveedor_adjudicado", "es_pyme", "es_grupo",
    "fecha_firma", "fecha_inicio", "fecha_fin", "ultima_actualizacion", "dias_adicionados",
    "valor_contrato", "valor_pago_adelantado", "valor_facturado", "valor_pagado",
    "valor_pendiente_pago", "valor_pendiente_ejecucion", "url_proceso",
]

df_silver = construir_silver(df_aceptados, COLUMNAS_SILVER)
df_dql = construir_dql(df_rechazados)

---
## Paso 3: Escritura idempotente (MERGE) en Silver y DQL

In [ ]:
%python
m_silver = escribir_merge(
    df_silver, SILVER_SECOP, ["id_contrato"], capa="silver",
    comentario="Silver - Contratos SECOP II validados y tipados: version vigente de cada contrato.",
)
m_dql = escribir_merge(
    df_dql, DQL_SECOP, ["_id_registro_bronze"], capa="dql", columna_fecha="_fecha_rechazo",
    comentario="DQL - Contratos SECOP rechazados, con la fila original de Bronze y el motivo.",
)

conteos = conciliar_fuente("SECOP")

---
## Paso 4: Validación

In [ ]:
%sql
SELECT _motivo_rechazo, COUNT(*) AS registros
FROM datos_empresas.dql.de_secop_contratos_dql
GROUP BY _motivo_rechazo
ORDER BY registros DESC;

In [ ]:
%sql
SELECT id_contrato, nombre_entidad, proveedor_adjudicado, modalidad_contratacion,
       valor_contrato, valor_pagado, fecha_firma, fecha_inicio, fecha_fin
FROM datos_empresas.silver.de_secop_contratos
LIMIT 10;

In [ ]:
%python
finalizar({"notebook": "04_Silver_SECOP", "conteos": conteos, "silver": m_silver, "dql": m_dql})